In [1]:
print(123)

123


In [2]:
#!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
#!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

In [3]:
from dotenv import load_dotenv
from openai import OpenAI
import os 
load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)


In [4]:
from ingest import load_faq_data, build_index

In [5]:
documents = load_faq_data()
index = build_index(documents)

In [6]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [7]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally, follow these steps:

1. Install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:
   - For **macOS**, download the `.pkg` and install it.
   - For **Windows**, download the `.msi` and install it.
   - For **Linux**, run the following command in the terminal:

   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. Once installed, open a terminal and type:

   ```bash
   ollama run llama3
   ```

   This command will:
   - Download the LLaMA 3 model (~4GB).
   - Start the model locally.
   - Open a chat-like interface where you can type questions.

3. To test the Ollama local server, run the following command:

   ```bash
   curl http://localhost:11434
   ```

   You should receive a response similar to:

   ```json
   {"models": [...]}
   ```


In [8]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = openai_client.responses.create(
    model='llama-3.1-8b-instant',
    input=messages,
)

response.output_text

"This conversation has just started, so I don't have any context about a specific course. I'm happy to chat with you and see if I can help you find the information you're looking for.\n\nCan you please tell me more about the course you're interested in joining? What's the name of the course, and where did you hear about it? I'll do my best to help you learn more and join it if possible."

In [9]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [10]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [20]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}


In [11]:
response = openai_client.responses.create(
    model='llama-3.1-8b-instant',
    input=messages,
    tools=[search_tool]
)

In [34]:
response = openai_client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=messages,
    tools=[search_tool]
)
#    model='llama-3.1-8b-instant',


In [35]:
len(response.choices)

1

In [36]:
print(response.choices[0].message)


ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='66hdcrq1e', function=Function(arguments='{"query":"joining the course"}', name='search'), type='function')])


In [40]:
#call = response.output[0]
call = response.choices[0].message.tool_calls[0]
args = json.loads(call.function.arguments)
args

{'query': 'joining the course'}

In [51]:
call

ChatCompletionMessageFunctionToolCall(id='66hdcrq1e', function=Function(arguments='{"query":"joining the course"}', name='search'), type='function')

In [46]:
call.function.name

'search'

In [45]:
results = seabrch(**args)
results

[{'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'How should I start the course and follow the weekly workflow?',
  'answer': 'Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\n\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nA typical workflow is:\n\n1. Watch the lesson videos.\n2. Work through the lesson notebooks/code.\n3. Read the homework instructions on GitHub.\n4. Submit answers through the course platform before the deadline.\n\nHomework is similar to the lesson flow, but uses a different dataset or slightly different task.',
  'doc_id': '04919992b3'},
 {'course': 'll

In [47]:
result_json = json.dumps(results, indent=2)

In [52]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.id,
    'output': result_json,
}

In [53]:
messages.append(call)

In [66]:
# messages.append(function_call_output)
messages.append(response.choices[0].message)
search_results = search(args['query'])

messages.append({
    'role': 'tool',
    'tool_call_id': call.id,
    'content': json.dumps(search_results)
})

In [67]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ChatCompletionMessageFunctionToolCall(id='66hdcrq1e', function=Function(arguments='{"query":"joining the course"}', name='search'), type='function'),
 {'type': 'function_call_output',
  'call_id': '66hdcrq1e',
  'output': '[\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\\n\\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\\n\\nA typical workflow is:\\n\\n1. 

In [68]:
response = openai_client.chat.completions.create(
    model='llama-3.3-70b-versatile',
    messages=messages,
    tools=[search_tool]
)


BadRequestError: Error code: 400 - {'error': {'message': "'messages.1' : input does not contain the discriminator property 'role'", 'type': 'invalid_request_error'}}

In [70]:
# Append assistant message as plain dict
messages.append(response.choices[0].message.model_dump())

# Append tool result
search_results = search(args['query'])


